# 🚲 Ćwiczenie 6: Window Functions — Wypożyczalnia rowerów miejskich

**Kurs:** Python w Zastosowaniach Big Data
**Zajęcia nr:** 6 (10.11.2026)
**Szacowany czas:** ⏱️ ~90 minut
**Poziom trudności:** ⭐⭐⭐ (Średni)

## 📚 Cele nauki

Po ukończeniu tego notebooka będziesz potrafić:
1. Definiować **window functions** przy pomocy `Window.partitionBy()` i `orderBy()`
2. Liczyć **rankingi** (`rank`, `row_number`) oraz **sumy narastające** (`rowsBetween`)
3. Używać `lag()`/`lead()` do analiz sekwencyjnych (poprzedni/następny przejazd)
4. Czytać plan zapytania (`.explain()`) i rozpoznawać shuffle (`Exchange`)
5. Porównać `repartition()` i `coalesce()` pod kątem wydajności

## 📋 Wymagania wstępne
- Znajomość DataFrame API i Spark SQL (Wykład 5)
- Podstawy `groupBy`/`agg` w PySpark

---
> 💡 **Tip:** Uruchamiaj komórki po kolei (`Shift+Enter`). Jeśli utkniesz, sprawdź hint (komórka z `@title`) albo zapytaj prowadzącego!

In [ ]:
# @title 🔧 Setup — uruchom tę komórkę jako pierwszą!

!pip install -q pyspark

import random
import time
from datetime import datetime, timedelta

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, TimestampType, DoubleType
)

random.seed(42)  # 👈 powtarzalność wyników

spark = (
    SparkSession.builder
    .appName("Cwiczenie6-RoweryMiejskie")
    .master("local[*]")
    .getOrCreate()
)

print("✅ Setup zakończony! Zaczynamy 🚀")

## 📖 Wprowadzenie — window functions w pigułce

`groupBy()` **redukuje** liczbę wierszy do jednego per grupa. **Window function** dodaje nową kolumnę obliczoną na podstawie "okna" powiązanych wierszy — bez utraty żadnego wiersza.

| Koncept | Opis | Przykład |
|---------|------|----------|
| `Window.partitionBy(col)` | Dzieli dane na grupy (jak `groupBy`, ale bez redukcji) | `Window.partitionBy("stacja")` |
| `Window.orderBy(col)` | Kolejność wierszy wewnątrz partycji | `.orderBy("czas_startu")` |
| `F.rank()` / `F.row_number()` | Ranking (z lukami / bez luk przy remisach) | `F.rank().over(okno)` |
| `F.lag()` / `F.lead()` | Wartość z poprzedniego / następnego wiersza | `F.lag("kolumna", 1).over(okno)` |
| `.rowsBetween(a, b)` | Zakres okna względem bieżącego wiersza | `rowsBetween(Window.unboundedPreceding, Window.currentRow)` |

Dziś pracujemy na **syntetycznym zbiorze ~200 000 przejazdów** rowerów miejskich w trzech polskich miastach 🚲🇵🇱.

In [ ]:
# @title 📊 Generowanie danych — wypożyczenia rowerów miejskich

CITIES = ["Warszawa", "Kraków", "Poznań"]
STATIONS_PER_CITY = 8
N_TRIPS = 200_000

stacje = [
    (miasto, f"{miasto}-Stacja-{i:02d}")
    for miasto in CITIES
    for i in range(1, STATIONS_PER_CITY + 1)
]
stacje_per_miasto = {
    miasto: [nazwa for m, nazwa in stacje if m == miasto] for miasto in CITIES
}

POCZATEK_OKRESU = datetime(2026, 8, 1)


def losowy_czas_startu() -> datetime:
    losowy_dzien = random.randint(0, 90)
    losowa_godzina = random.randint(5, 23)
    losowa_minuta = random.randint(0, 59)
    return POCZATEK_OKRESU + timedelta(
        days=losowy_dzien, hours=losowa_godzina, minutes=losowa_minuta
    )


wiersze = []
for trip_id in range(1, N_TRIPS + 1):
    miasto = random.choice(CITIES)
    stacja_startowa = random.choice(stacje_per_miasto[miasto])
    stacja_koncowa = random.choice(stacje_per_miasto[miasto])

    dystans_km = round(random.uniform(0.3, 12.0), 2)
    predkosc_kmh = random.uniform(9.0, 16.0)
    szum_min = random.uniform(-2.0, 5.0)
    czas_trwania_min = max(1.0, round(dystans_km / predkosc_kmh * 60 + szum_min, 1))

    czas_startu = losowy_czas_startu()
    czas_konca = czas_startu + timedelta(minutes=czas_trwania_min)

    typ_uzytkownika = random.choices(["abonament", "jednorazowy"], weights=[0.7, 0.3])[0]

    wiersze.append((
        trip_id, miasto, stacja_startowa, stacja_koncowa,
        czas_startu, czas_konca, czas_trwania_min,
        typ_uzytkownika, dystans_km,
    ))

print(f"Wygenerowano {len(wiersze):,} przejazdów")

schema = StructType([
    StructField("trip_id", IntegerType(), False),
    StructField("city", StringType(), False),
    StructField("start_station", StringType(), False),
    StructField("end_station", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("duration_min", DoubleType(), False),
    StructField("user_type", StringType(), False),
    StructField("distance_km", DoubleType(), False),
])

trips = spark.createDataFrame(wiersze, schema=schema)
trips = trips.withColumn("trip_date", F.to_date("start_time"))
trips.cache()

print(f"📊 Liczba wierszy: {trips.count():,}")
trips.printSchema()
trips.show(5, truncate=False)

## 🎓 Przykład prowadzony — ranking stacji wg liczby przejazdów

**Zadanie:** Dla każdego miasta chcemy ranking stacji startowych wg liczby wypożyczeń — stacja #1 to najbardziej ruchliwa stacja w danym mieście.

In [ ]:
# @title 🎓 Przykład prowadzony — uruchom i przeanalizuj

# Krok 1: liczba przejazdów per stacja startowa (w ramach miasta)
wolumen_stacji = trips.groupBy("city", "start_station").agg(
    F.count("trip_id").alias("liczba_przejazdow")
)

# Krok 2: okno — partycja per miasto, sortowanie malejąco wg liczby przejazdów
okno_rankingu = Window.partitionBy("city").orderBy(F.desc("liczba_przejazdow"))

# Krok 3: nadanie rankingu
ranking_stacji = wolumen_stacji.withColumn(
    "ranking", F.rank().over(okno_rankingu)  # 👈 window function w akcji!
)

ranking_stacji.orderBy("city", "ranking").show(24, truncate=False)

---
## ✏️ Ćwiczenie 1: Ranking miast wg łącznej liczby przejazdów

**Trudność:** ⭐

Policz łączną liczbę przejazdów w każdym mieście, a następnie nadaj każdemu miastu **ranking** (bez `partitionBy` — jedno globalne okno!). Miasto z największą liczbą przejazdów powinno mieć `ranking == 1`.

> 💡 **Hint:** `Window.orderBy(...)` bez `partitionBy` tworzy jedno okno obejmujące cały DataFrame.

In [ ]:
# ✏️ Ćwiczenie 1: Ranking miast

liczba_przejazdow_miasto = trips.groupBy("city").agg(
    F.count("trip_id").alias("liczba_przejazdow")
)

# TODO: zdefiniuj okno sortujące malejąco wg liczby przejazdów (bez partitionBy)
okno_miast = ...

# TODO: nadaj ranking używając F.rank()
ranking_miast = liczba_przejazdow_miasto.withColumn(
    "ranking", ...
)

ranking_miast.orderBy("ranking").show()

In [ ]:
# @title ✅ Test — Ćwiczenie 1

try:
    assert "ranking" in ranking_miast.columns, "❌ Brakuje kolumny 'ranking'"
    wynik = ranking_miast.orderBy("ranking").collect()
    assert len(wynik) == 3, f"❌ Oczekiwano 3 miast, jest {len(wynik)}"
    assert wynik[0]["ranking"] == 1, "❌ Pierwsze miejsce powinno mieć ranking 1"
    print("✅ Świetnie! Ranking miast gotowy 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz 💪")

---
## ✏️ Ćwiczenie 2: Suma narastająca (running total) przejazdów per stacja

**Trudność:** ⭐⭐

Dla każdej stacji policz **dzienną** liczbę przejazdów, a następnie dodaj kolumnę z **sumą narastającą** (running total) w czasie — używając `rowsBetween(Window.unboundedPreceding, Window.currentRow)`.

**Oczekiwany wynik:** Kolumna `suma_narastajaca` powinna **nigdy nie maleć** w obrębie jednej stacji, a jej ostatnia wartość powinna równać się sumie wszystkich dziennych przejazdów tej stacji.

In [ ]:
# ✏️ Ćwiczenie 2: Running total

dzienne_przejazdy = trips.groupBy("start_station", "trip_date").agg(
    F.count("trip_id").alias("liczba_przejazdow_dziennie")
)

# TODO: zdefiniuj okno: partycja per stacja, sortowanie po dacie, zakres od początku do bieżącego wiersza
okno_narastajace = (
    Window.partitionBy("start_station")
    .orderBy("trip_date")
    .rowsBetween(...)  # TODO: (Window.unboundedPreceding, Window.currentRow)
)

# TODO: dodaj kolumnę z sumą narastającą, używając F.sum(...).over(...)
dzienne_z_suma = dzienne_przejazdy.withColumn(
    "suma_narastajaca", ...
)

dzienne_z_suma.orderBy("start_station", "trip_date").show(10)

In [ ]:
# @title ✅ Test — Ćwiczenie 2

try:
    przykladowa_stacja = dzienne_z_suma.select("start_station").first()["start_station"]
    pdf = (
        dzienne_z_suma.filter(F.col("start_station") == przykladowa_stacja)
        .orderBy("trip_date")
        .toPandas()
    )
    assert pdf["suma_narastajaca"].is_monotonic_increasing, (
        "❌ Suma narastająca powinna rosnąć (lub być stała) w czasie"
    )
    assert pdf["suma_narastajaca"].iloc[-1] == pdf["liczba_przejazdow_dziennie"].sum(), (
        "❌ Ostatnia wartość powinna równać się sumie wszystkich dni tej stacji"
    )
    print("✅ Running total działa poprawnie! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz 💪")

---
## ✏️ Ćwiczenie 3: Przerwa czasowa między kolejnymi przejazdami (`lag`)

**Trudność:** ⭐⭐

Dla każdej stacji startowej policz, ile minut minęło od **poprzedniego** przejazdu rozpoczętego z tej samej stacji (`lag`). Pierwszy przejazd ze stacji nie ma poprzednika — jego przerwa powinna być `null`.

> 💡 **Hint:** Odejmij `unix timestamp` obu kolumn (`.cast("long")`) i podziel przez 60, żeby dostać minuty.

In [ ]:
# ✏️ Ćwiczenie 3: lag — przerwa czasowa

okno_czasowe = Window.partitionBy("start_station").orderBy("start_time")

# TODO: dodaj kolumnę 'poprzedni_start' z czasem poprzedniego przejazdu (F.lag)
przejazdy_z_przerwa = trips.withColumn(
    "poprzedni_start", ...
)

przejazdy_z_przerwa = przejazdy_z_przerwa.withColumn(
    "przerwa_minuty",
    (F.col("start_time").cast("long") - F.col("poprzedni_start").cast("long")) / 60
)

przejazdy_z_przerwa.select(
    "start_station", "start_time", "poprzedni_start", "przerwa_minuty"
).orderBy("start_station", "start_time").show(10)

In [ ]:
# @title ✅ Test — Ćwiczenie 3

try:
    pierwszy_wiersz = przejazdy_z_przerwa.orderBy("start_station", "start_time").first()
    assert pierwszy_wiersz["poprzedni_start"] is None, (
        "❌ Pierwszy przejazd ze stacji nie powinien mieć poprzednika (oczekiwano null)"
    )
    probka = przejazdy_z_przerwa.filter(F.col("poprzedni_start").isNotNull()).limit(1000).collect()
    assert all(w["przerwa_minuty"] >= 0 for w in probka), "❌ Przerwa czasowa nie może być ujemna"
    print("✅ Obliczono przerwy czasowe poprawnie! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz 💪")

---
## ✏️ Ćwiczenie 4: Top-3 najbardziej ruchliwe stacje końcowe per miasto

**Trudność:** ⭐⭐

Znajdź **dokładnie 3** stacje końcowe (`end_station`) z największą liczbą przyjazdów w każdym mieście. Użyj `F.row_number()` — gwarantuje dokładnie 3 wiersze na miasto, nawet przy remisach.

> ⚠️ Gdybyś użył/a `F.rank()`, przy remisie mógłbyś dostać więcej niż 3 wiersze!

In [ ]:
# ✏️ Ćwiczenie 4: Top-3 stacje końcowe per miasto

liczba_przyjazdow = trips.groupBy("city", "end_station").agg(
    F.count("trip_id").alias("liczba_przyjazdow")
)

okno_top = Window.partitionBy("city").orderBy(F.desc("liczba_przyjazdow"))

# TODO: dodaj kolumnę 'pozycja' używając F.row_number()
top3 = liczba_przyjazdow.withColumn(
    "pozycja", ...
)

# TODO: przefiltruj tylko wiersze z pozycja <= 3
top3 = top3.filter(...)

top3.orderBy("city", "pozycja").show(9, truncate=False)

In [ ]:
# @title ✅ Test — Ćwiczenie 4

try:
    wyniki_per_miasto = top3.groupBy("city").count().collect()
    assert all(row["count"] == 3 for row in wyniki_per_miasto), (
        "❌ Każde miasto powinno mieć dokładnie 3 stacje w top3"
    )
    pierwsze_miasto = CITIES[0]
    pozycje = sorted(
        row["pozycja"] for row in top3.filter(F.col("city") == pierwsze_miasto).collect()
    )
    assert pozycje == [1, 2, 3], f"❌ Oczekiwano pozycji [1, 2, 3], jest {pozycje}"
    print("✅ Top-3 stacje per miasto obliczone poprawnie! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz 💪")

---
## ✏️ Ćwiczenie 5: Średnia krocząca czasu trwania przejazdu

**Trudność:** ⭐⭐⭐

Dla każdej stacji startowej policz **średnią kroczącą** (moving average) `duration_min` z bieżącego i 2 poprzednich przejazdów (`rowsBetween(-2, 0)`), posortowanych po `start_time`.

**Oczekiwany wynik:** Dla pierwszego przejazdu ze stacji średnia krocząca powinna równać się jego własnemu `duration_min` (bo nie ma jeszcze 2 poprzednich).

In [ ]:
# ✏️ Ćwiczenie 5: Średnia krocząca

# TODO: zdefiniuj okno z rowsBetween(-2, 0) — bieżący wiersz + 2 poprzednie
okno_kroczace = (
    Window.partitionBy("start_station")
    .orderBy("start_time")
    .rowsBetween(...)
)

# TODO: dodaj kolumnę 'srednia_kroczaca_min' używając F.avg("duration_min").over(...)
z_srednia_kroczaca = trips.withColumn(
    "srednia_kroczaca_min", ...
)

z_srednia_kroczaca.select(
    "start_station", "start_time", "duration_min", "srednia_kroczaca_min"
).orderBy("start_station", "start_time").show(10)

In [ ]:
# @title ✅ Test — Ćwiczenie 5

try:
    pierwszy = z_srednia_kroczaca.orderBy("start_station", "start_time").first()
    assert abs(pierwszy["srednia_kroczaca_min"] - pierwszy["duration_min"]) < 0.01, (
        "❌ Dla pierwszego przejazdu średnia krocząca powinna równać się jego własnemu czasowi trwania"
    )
    print("✅ Średnia krocząca policzona poprawnie! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz 💪")

---
## ✏️ Ćwiczenie 6: Czytanie planu zapytania (`.explain()`)

**Trudność:** ⭐⭐

Uruchom poniższe zapytanie i przeanalizuj wypisany `Physical Plan`. Na tej podstawie odpowiedz (wpisując `True`/`False`) na dwa pytania.

> 💡 **Hint:** Shuffle w planie fizycznym objawia się węzłem `Exchange`. Predicate pushdown oznacza, że `Filter` pojawia się **przed** (czyli fizycznie bliżej źródła danych niż) `Aggregate`/`HashAggregate`.

In [ ]:
# ✏️ Ćwiczenie 6: Czytanie planu zapytania

zapytanie = (
    trips.filter(F.col("user_type") == "abonament")
    .groupBy("city")
    .agg(F.sum("distance_km").alias("suma_dystansu"))
    .orderBy(F.desc("suma_dystansu"))
)

zapytanie.explain(True)

# TODO: na podstawie wypisanego Physical Plan odpowiedz True/False
czy_jest_shuffle = ...  # Czy w Physical Plan widzisz węzeł "Exchange"?
czy_filtr_jest_wczesnie = ...  # Czy "Filter" pojawia się przed "HashAggregate" w planie fizycznym?

In [ ]:
# @title ✅ Test — Ćwiczenie 6

try:
    assert czy_jest_shuffle is True, (
        "❌ groupBy + orderBy generują shuffle — sprawdź plan jeszcze raz (szukaj 'Exchange')"
    )
    assert czy_filtr_jest_wczesnie is True, (
        "❌ Catalyst powinien zepchnąć Filter przed Aggregate (predicate pushdown)"
    )
    print("✅ Dobrze przeczytany plan zapytania! 🎉")
except AssertionError as e:
    print(str(e))
    print("Uruchom jeszcze raz zapytanie.explain(True) i przyjrzyj się dokładnie 💪")

---
## ✏️ Ćwiczenie 7: `repartition()` vs `coalesce()` — porównanie czasu

**Trudność:** ⭐⭐⭐

Zmierz czas wykonania zmniejszenia liczby partycji do 4, używając osobno `repartition(4)` i `coalesce(4)`. Zapisz zmierzone czasy i liczby partycji po każdej operacji.

> 💡 **Hint:** `repartition()` zawsze robi pełny shuffle (drożej), `coalesce()` łączy sąsiadujące partycje bez pełnego shuffla (zwykle taniej).

In [ ]:
# ✏️ Ćwiczenie 7: repartition vs coalesce

print(f"Liczba partycji na starcie: {trips.rdd.getNumPartitions()}")

start = time.time()
# TODO: użyj trips.repartition(4)
trips_repartition = ...
trips_repartition.count()  # akcja wymuszająca wykonanie
czas_repartition = time.time() - start

start = time.time()
# TODO: użyj trips.coalesce(4)
trips_coalesce = ...
trips_coalesce.count()
czas_coalesce = time.time() - start

print(f"repartition(4): {czas_repartition:.2f}s")
print(f"coalesce(4):    {czas_coalesce:.2f}s")

# TODO: uzupełnij liczbę partycji po każdej operacji
liczba_partycji_repartition = ...
liczba_partycji_coalesce = ...

In [ ]:
# @title ✅ Test — Ćwiczenie 7

try:
    assert liczba_partycji_repartition == 4, "❌ repartition(4) powinno dać 4 partycje"
    assert liczba_partycji_coalesce == 4, "❌ coalesce(4) powinno dać 4 partycje"
    print("✅ Sprawdzone! coalesce zwykle jest szybszy przy zmniejszaniu partycji, bo unika pełnego shuffla 🚀")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz 💪")

---
## 🏆 Bonus Challenge: Wykrywanie potencjalnego "rebalancingu" rowerów

Operator wypożyczalni czasem ręcznie **przewozi rowery** między stacjami ("rebalancing"), żeby np. uzupełnić pustą stację w centrum. Możemy to pośrednio wykryć: jeśli między **przyjazdem** roweru na stację a **kolejnym odjazdem** z tej samej stacji mija bardzo dużo czasu (> 240 minut), to podejrzanie długo ten rower tam "stał" — a scenariusz interwencji operatora jest prawdopodobny.

**Zadanie:**
1. Zbuduj DataFrame zdarzeń łączący przyjazdy (`end_station`, `end_time`) i odjazdy (`start_station`, `start_time`) dla każdej stacji
2. Użyj `F.lead()`, żeby dla każdego zdarzenia znać **następne** zdarzenie na tej samej stacji
3. Oznacz jako anomalię wiersze, gdzie: bieżące zdarzenie to `"przyjazd"`, następne to `"odjazd"`, a przerwa > 240 minut

In [ ]:
# 🏆 Bonus: wykrywanie anomalii rebalancingu

przyjazdy = trips.select(
    F.col("end_station").alias("stacja"),
    F.col("end_time").alias("czas_zdarzenia"),
    F.lit("przyjazd").alias("typ_zdarzenia"),
)
odjazdy = trips.select(
    F.col("start_station").alias("stacja"),
    F.col("start_time").alias("czas_zdarzenia"),
    F.lit("odjazd").alias("typ_zdarzenia"),
)
zdarzenia = przyjazdy.unionByName(odjazdy)

okno_zdarzen = Window.partitionBy("stacja").orderBy("czas_zdarzenia")

zdarzenia_z_nastepnym = (
    zdarzenia
    .withColumn("nastepny_czas", F.lead("czas_zdarzenia", 1).over(okno_zdarzen))
    .withColumn("nastepny_typ", F.lead("typ_zdarzenia", 1).over(okno_zdarzen))
    .withColumn(
        "przerwa_minut",
        (F.col("nastepny_czas").cast("long") - F.col("czas_zdarzenia").cast("long")) / 60
    )
)

# TODO: dopisz warunek filtrowania — przyjazd -> odjazd z przerwą > 240 minut
anomalie_rebalancingu = zdarzenia_z_nastepnym.filter(
    ...
)

anomalie_rebalancingu.select(
    "stacja", "czas_zdarzenia", "nastepny_czas", "przerwa_minut"
).orderBy(F.desc("przerwa_minut")).show(10)

In [ ]:
# @title ✅ Test — Bonus

try:
    kolumny_wymagane = {
        "stacja", "czas_zdarzenia", "typ_zdarzenia",
        "nastepny_czas", "nastepny_typ", "przerwa_minut",
    }
    assert kolumny_wymagane.issubset(set(anomalie_rebalancingu.columns)), (
        "❌ Brakuje wymaganych kolumn w wyniku"
    )
    liczba_anomalii = anomalie_rebalancingu.count()
    if liczba_anomalii > 0:
        przyklad = anomalie_rebalancingu.first()
        assert przyklad["przerwa_minut"] > 240, "❌ Przerwa powinna przekraczać próg 240 minut"
        assert przyklad["typ_zdarzenia"] == "przyjazd", "❌ Bieżące zdarzenie powinno być przyjazdem"
        assert przyklad["nastepny_typ"] == "odjazd", "❌ Następne zdarzenie powinno być odjazdem"
    print(f"🏆 Bonus zaliczony! Znaleziono {liczba_anomalii} potencjalnych zdarzeń rebalancingu")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz 💪")

---
## 📝 Podsumowanie

Świetna robota! 🎉 W tym ćwiczeniu przećwiczyłeś/aś:

- ✅ Definiowanie okien (`Window.partitionBy()` + `orderBy()`)
- ✅ Rankingi (`rank`, `row_number`) i różnicę między nimi przy remisach
- ✅ Sumy narastające i średnie kroczące (`rowsBetween`)
- ✅ Analizy sekwencyjne "poprzedni/następny" (`lag`, `lead`)
- ✅ Czytanie planu zapytania (`.explain()`) i rozpoznawanie shuffle
- ✅ Różnicę między `repartition()` a `coalesce()` w praktyce

### 📚 Dalsze materiały
- [Window Functions API (PySpark)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/window.html)
- [Catalyst Optimizer — Deep Dive (Databricks)](https://databricks.com/blog/2015/04/13/deep-dive-into-spark-sqls-catalyst-optimizer.html)

### 🔮 Co dalej?
Na **Wykładzie 7** przechodzimy do przygotowania danych do Machine Learning — **Feature Engineering**! Te same dane rowerowe posłużą nam jako materiał do ćwiczeń.